# Week 9 Companion Notebook: Spark & Distributed Transformation

## ISM 6562 — Big Data for Business Applications

---

This notebook is your hands-on companion to the Week 9 lecture slides on **Apache Spark and the Transform layer** of the ELT pipeline. Every section below maps to a specific section of the lecture deck, so you can work through them in parallel or jump to whichever one matches what you are studying.

| Notebook Part | Lecture Section | What you will do |
|---|---|---|
| **Part 0 — Setup** | §3 Architecture | Start a SparkSession, download data, map the running cluster to the slide diagrams |
| **Part 1 — Spark Fundamentals** | §4 Spark Programming | DataFrames, transformations, SQL, window functions, lazy evaluation, reading the DAG |
| **Part 2 — The "T" in ELT** | §5 The "T" in ELT | A complete three-stage pipeline: ingest → curate → analytics |
| **Part 3 — Optimization (optional deep dive)** | §6 Optimization | Caching, query plans, partition pruning, broadcast joins, repartitioning |
| **Part 4 — HealthPulse** | §7 HealthPulse Transforms | Schema harmonization across four hospitals, claims enrichment, device anomaly detection, lab result analysis |
| **Part 5 — GreenRoute** | §8 GreenRoute Transforms | Partitioned Avro reads, trip segmentation with window functions, fuel efficiency, weather joins |
| **Part 6 — Reflection** | §9 Summary | Questions to connect the hands-on work back to the architectural concepts |

The notebook is designed to run **end to end** against the Week 9 Docker stack. Every cell that takes more than a few seconds mentions it explicitly, and every output has a short "what you should see" comment so you can check your work.


---

## Part 0: Setup & Cluster Map

### What is running when this notebook opens

The `docker-compose.yml` for this week brings up **four containers** on a shared Docker network. Each plays one of the Spark roles you saw in the lecture diagrams (§3, slides 20–22, 24):

| Container | Port → host | Spark role (see lecture slide 24) | UI |
|---|---|---|---|
| `spark-master` | 8080, 7077, 4040 | **Cluster Manager** — allocates cores & memory to apps | `http://localhost:8080` |
| `spark-worker-1` | 8081 | **Executor JVM #1** — 2 cores, 2 GB | `http://localhost:8081` |
| `spark-worker-2` | 8082 | **Executor JVM #2** — 2 cores, 2 GB | `http://localhost:8082` |
| `spark-jupyter` | 8888 | **Driver** — this notebook's kernel IS the Spark Driver | `http://localhost:8888` |

When you run the next cell, the notebook kernel creates a `SparkSession`, and from that moment on:

- **This notebook process** is the Driver (slide 21).
- The two `spark-worker-*` containers spin up one Executor each on your behalf (slide 22).
- `spark-master` hands out the cores it has to your application.
- A temporary **Application UI** appears at `http://localhost:4040` that lives only as long as this kernel stays alive (slide 33b — the DAG lives there).

If you have the cluster running, open **all four UIs in browser tabs now** and keep them visible while you work through the rest of the notebook. You will refer back to them repeatedly.

### Starting the stack

If you have not already brought it up, in a terminal from the `week09-spark-distributed-transformation/` folder:

```bash
docker compose up -d
```

Then open Jupyter at `http://localhost:8888?token=spark` and start running the cells below.


### 0.1 — Create the SparkSession (matches slide 28)

Every PySpark app starts by creating a `SparkSession`. The lines below map one-to-one to the annotated slide 28 example, so you know what each `.config(...)` does.


In [ ]:
# SparkSession.builder is the fluent API for configuring the session;
# each .config(...) sets one cluster-side parameter BEFORE the session starts.
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("Week9-CompanionNotebook")           # label visible in Spark Master UI / :4040
    .master("spark://spark-master:7077")          # cluster manager (Spark Standalone)
    .config("spark.executor.memory", "1g")        # RAM per Executor JVM
    .config("spark.driver.memory",   "1g")        # RAM this notebook kernel gets as Driver
    .config("spark.sql.shuffle.partitions", "8")  # # of partitions AFTER a shuffle (join / groupBy).
                                                   # Default is 200 — too many for a 4-core cluster;
                                                   # 8 gives each of our 4 cores 2 tasks to do.
    .config("spark.jars.packages",
            "org.apache.spark:spark-avro_2.12:3.5.0")  # Avro connector for GreenRoute GPS
    # The Driver (this Jupyter kernel) runs Python 3.11; the workers have Python 3.11
    # installed via miniconda at /opt/miniconda/bin/python3. Spark pickles data between
    # the two and requires matching minor versions — tell Spark which Python to invoke
    # on each side explicitly so there is no surprise.
    .config("spark.pyspark.python",        "/opt/miniconda/bin/python3")
    .config("spark.pyspark.driver.python", "/opt/conda/bin/python3")
    .getOrCreate())                                # reuse an existing session if present

# Verify the connection
print(f"Spark version:        {spark.version}")
print(f"Default parallelism:  {spark.sparkContext.defaultParallelism}  "
      f"(this equals total executor cores across the cluster)")
print(f"Application ID:       {spark.sparkContext.applicationId}")
print(f"Master URL:           {spark.sparkContext.master}")

print()
print("Now open the Spark UIs in new browser tabs:")
print("   http://localhost:8080   — Spark Master (cluster overview)")
print("   http://localhost:4040   — Application UI (will populate as you run cells below)")


### 0.2 — Download the data files

All datasets used in this notebook live in the public class repo. The cell below downloads them into `/home/jovyan/data` (a path mounted into all four containers at the same location, so Driver and Executors can both see it). Re-running the cell is safe — it skips files already on disk.

The first run takes ~30 seconds depending on your connection. You should see **84 files** at the end: patient CSVs, device JSON, insurance CSV, lab JSON, delivery JSON, weather JSON, route plans JSON, fuel CSV, and 72 GPS Avro partitions (3 days × 24 hours).


In [ ]:
# Download all data files from the ism6562s26-class public repo
import os, subprocess

REPO = "https://raw.githubusercontent.com/prof-tcsmith/ism6562s26-class/main"
DATA_DIR = "/home/jovyan/data"

files_to_download = [
    # Week 8 HealthPulse data
    ("week08/data/healthpulse/patient-records/hospital_01_patients.csv", "healthpulse/patient-records/hospital_01_patients.csv"),
    ("week08/data/healthpulse/patient-records/hospital_02_patients.csv", "healthpulse/patient-records/hospital_02_patients.csv"),
    ("week08/data/healthpulse/patient-records/hospital_03_patients.csv", "healthpulse/patient-records/hospital_03_patients.csv"),
    ("week08/data/healthpulse/device-readings/device_readings.json",     "healthpulse/device-readings/device_readings.json"),
    ("week08/data/healthpulse/insurance-claims/insurance_claims.csv",    "healthpulse/insurance-claims/insurance_claims.csv"),
    # Week 9 HealthPulse additions
    ("week09/data/healthpulse/patient-records/hospital_04_patients.csv", "healthpulse/patient-records/hospital_04_patients.csv"),
    ("week09/data/healthpulse/lab-results/lab_results.json",             "healthpulse/lab-results/lab_results.json"),
    # Week 8 GreenRoute data
    ("week08/data/greenroute/delivery-receipts/delivery_receipts.json",  "greenroute/delivery-receipts/delivery_receipts.json"),
    ("week08/data/greenroute/weather/weather_data.json",                 "greenroute/weather/weather_data.json"),
    ("week08/data/greenroute/gps-telemetry/gps_reading.avsc",            "greenroute/gps-telemetry/gps_reading.avsc"),
    # Week 9 GreenRoute additions
    ("week09/data/greenroute/route-plans/route_plans.json",              "greenroute/route-plans/route_plans.json"),
    ("week09/data/greenroute/fuel-logs/fuel_logs.csv",                   "greenroute/fuel-logs/fuel_logs.csv"),
]

# 3 days × 24 hours of GPS telemetry (Avro)
for day in range(1, 4):
    for hour in range(24):
        remote = f"week08/data/greenroute/gps-telemetry/day_{day:02d}/hour_{hour:02d}/gps_data.avro"
        local  = f"greenroute/gps-telemetry/day_{day:02d}/hour_{hour:02d}/gps_data.avro"
        files_to_download.append((remote, local))

downloaded = skipped = 0
for remote_path, local_path in files_to_download:
    full_local = os.path.join(DATA_DIR, local_path)
    os.makedirs(os.path.dirname(full_local), exist_ok=True)
    if os.path.exists(full_local):
        skipped += 1
        continue
    result = subprocess.run(
        ["curl", "-sL", f"{REPO}/{remote_path}", "-o", full_local],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        downloaded += 1
    else:
        print(f"  WARNING: failed to download {remote_path}")

print(f"Download complete: {downloaded} new, {skipped} already present, {downloaded + skipped} total")


In [ ]:
# Quick inventory of what was downloaded
import os, glob

print("Top-level datasets:")
for name in ["healthpulse/patient-records",
             "healthpulse/device-readings",
             "healthpulse/insurance-claims",
             "healthpulse/lab-results",
             "greenroute/delivery-receipts",
             "greenroute/weather",
             "greenroute/route-plans",
             "greenroute/fuel-logs"]:
    path = f"{DATA_DIR}/{name}"
    n_files = sum(1 for _ in os.scandir(path)) if os.path.exists(path) else 0
    size_mb = sum(os.path.getsize(os.path.join(r, f))
                  for r, _, fs in os.walk(path) for f in fs) / 1024 / 1024
    print(f"   {name:<40s}  {n_files:3d} file(s), {size_mb:6.2f} MB")

gps_files = glob.glob(f"{DATA_DIR}/greenroute/gps-telemetry/day_*/hour_*/gps_data.avro")
gps_size = sum(os.path.getsize(f) for f in gps_files) / 1024 / 1024
print(f"   greenroute/gps-telemetry (Avro)           {len(gps_files):3d} file(s), {gps_size:6.2f} MB")


---

## Part 1: Spark Fundamentals

**Maps to lecture §4 (Spark Programming with PySpark), slides 27–36.**

This part walks through every core DataFrame operation you saw on the slides: reading data, transforming it, joining it, aggregating it, running window functions, querying it with Spark SQL, and reading the DAG the whole thing produced.


### 1a. Reading data — CSV, JSON, Parquet

The slides (§4, slide 29) compared the three main file formats you will encounter in the data lake. Let's read one of each and inspect what Spark gives us back.


In [ ]:
# CSV — text format, row-oriented. Requires header=True and inferSchema=True
# to get proper column names and types. Schema inference triggers a full file
# scan on read, so for big data the inferred schema is a one-time cost.
patients_h01 = spark.read.csv(
    "/home/jovyan/data/healthpulse/patient-records/hospital_01_patients.csv",
    header=True,
    inferSchema=True,
)

print("=== Hospital 01 Patients (CSV) ===")
print(f"Rows: {patients_h01.count()}, Columns: {len(patients_h01.columns)}")
patients_h01.printSchema()
patients_h01.show(5, truncate=False)


In [ ]:
# JSON — text format, row-oriented, self-describing (nested structs are native).
# Spark infers the schema automatically; no need for header/inferSchema flags.
# Notice how Spark represents the nested `reading` field as a struct type.
device_readings = spark.read.json(
    "/home/jovyan/data/healthpulse/device-readings/device_readings.json"
)

print("=== Device Readings (JSON, nested) ===")
print(f"Rows: {device_readings.count()}, Columns: {len(device_readings.columns)}")
device_readings.printSchema()
device_readings.show(3, truncate=False)


**Parquet** will show up later in the notebook (Part 2 and Part 4) where we WRITE it. When you read Parquet, there is no schema inference — the schema is embedded in the file metadata. Parquet is also columnar, so Spark only reads the columns your query mentions. This is why it is the standard format for curated data lake zones.


### 1b. Core transformations — filter, select, withColumn, when

Transformations are **lazy** — they define a new DataFrame without executing anything. The first lines of the lecture cheat sheet (§4, slide 30) live right here.

One gotcha: the CSV has `dob` (date of birth), not `age`. So step 1 is to derive `age` with `withColumn` — and from there the DataFrame becomes the starting point for all the examples below. The comment in the slide material about "pediatric/adult/senior" uses `when/otherwise` over this same derived column.


In [ ]:
from pyspark.sql import functions as F

# withColumn — add or replace a column. Here we derive age from dob.
patients_h01 = patients_h01.withColumn(
    "age",
    F.floor(F.datediff(F.current_date(), F.col("dob")) / 365.25).cast("int"),
)
patients_h01.select("patient_id", "dob", "age").show(5)


In [ ]:
# select — project a subset of columns
patients_h01.select("patient_id", "name", "age", "dept").show(5)

# filter — keep rows matching a condition (pure SQL WHERE)
elderly = patients_h01.filter(F.col("age") >= 65)
print(f"Patients aged 65+: {elderly.count()}")

# when/otherwise — SQL CASE expression over a column
banded = patients_h01.withColumn(
    "age_group",
    F.when(F.col("age") < 18, "Pediatric")
     .when(F.col("age") < 65, "Adult")
     .otherwise("Senior"),
)
banded.select("patient_id", "age", "age_group").show(10)


### 1c. Aggregations — groupBy + agg

`groupBy(keys).agg(...)` collapses rows into one-per-key with whatever aggregate expressions you list. This is the workhorse of descriptive analytics.


In [ ]:
# One row per department, four aggregate columns
dept_stats = (patients_h01.groupBy("dept")
    .agg(
        F.count("*").alias("patient_count"),
        F.round(F.avg("age"), 1).alias("avg_age"),
        F.min("age").alias("min_age"),
        F.max("age").alias("max_age"),
    )
    .orderBy(F.desc("patient_count")))

dept_stats.show(truncate=False)


### 1d. Joins — inner, left, broadcast

The slides (§4, slide 31) compared join types and introduced broadcast joins. Let's reproduce the broadcast-join example here and read its physical plan.


In [ ]:
claims = spark.read.csv(
    "/home/jovyan/data/healthpulse/insurance-claims/insurance_claims.csv",
    header=True, inferSchema=True,
)
print(f"Insurance claims: {claims.count():,} rows")
claims.printSchema()
# Note the real column names: billed_amount (not amount), service_date (not claim_date),
# status in {'approved','denied',...}. We'll use these names throughout.

# Inner join on patient_id — only patients with at least one claim survive
patient_claims = patients_h01.join(claims, on="patient_id", how="inner")
print(f"\nPatients ⨯ Claims: {patient_claims.count():,} rows")
patient_claims.select("patient_id", "name", "dept", "service_date", "billed_amount", "status").show(5, truncate=False)


In [ ]:
# Broadcast join — ship the small table to every Executor, join locally against
# each partition of the big table. No shuffle of the big side.
#
# This tiny in-memory DataFrame is the "small side" — Spark would auto-broadcast
# it because it is under the default 10 MB threshold, but the explicit hint
# documents intent and guarantees the optimization.
dept_lookup = spark.createDataFrame([
    ("Cardiology",   "Heart and cardiovascular system"),
    ("Neurology",    "Brain and nervous system"),
    ("Oncology",     "Cancer treatment"),
    ("Pediatrics",   "Children's health"),
    ("Orthopedics",  "Bones and joints"),
    ("Emergency",    "Emergency medicine"),
], ["dept", "description"])

enriched = patients_h01.join(F.broadcast(dept_lookup), on="dept", how="left")
print("=== Physical plan (look for 'BroadcastHashJoin' — not 'SortMergeJoin') ===")
enriched.explain()

enriched.select("patient_id", "dept", "description").show(5, truncate=False)


### 1e. Window functions — the same idea as SQL `OVER(...)`

This section matches slide 32 in the deck. A **window function** computes a value for each row in relation to a group of rows, **without** collapsing the group into one row (the way `groupBy` does). Anyone who has written `OVER (PARTITION BY ... ORDER BY ...)` in Postgres has used windows before — the DataFrame API below is just a programmatic wrapper over exactly that SQL construct.

> ⚠ **Different concept, same word.** "Window" in *this* section means a **relational window** (a frame of rows in a result set). Later, in streaming (Week 10), "window" will mean a **time window** (tumbling / sliding / session). Same word, two different concepts.


In [ ]:
from pyspark.sql.window import Window

# Build a tiny orders DataFrame in memory so the window examples are self-contained
# and easy to eyeball. Each row is one order, with a customer, date, and amount.
orders = spark.createDataFrame([
    ("c1", "2025-01-05", 120.0), ("c1", "2025-01-12",  75.0), ("c1", "2025-02-03", 200.0),
    ("c1", "2025-02-28",  40.0), ("c1", "2025-03-15",  90.0),
    ("c2", "2025-01-07",  55.0), ("c2", "2025-02-15", 150.0), ("c2", "2025-03-02", 300.0),
    ("c3", "2025-01-20",  30.0), ("c3", "2025-02-10",  60.0), ("c3", "2025-03-05",  45.0),
], ["customer_id", "order_date", "amount"]).withColumn(
    "order_date", F.to_date("order_date")
)
orders.show()


In [ ]:
# Define the window: for each customer, look at their orders in date order.
# This is the SQL equivalent of  OVER (PARTITION BY customer_id ORDER BY order_date)
w = Window.partitionBy("customer_id").orderBy("order_date")

enriched = (orders
    .withColumn("running_total",
                F.round(F.sum("amount").over(w), 2))     # SUM(amount) OVER (...)
    .withColumn("order_number",
                F.row_number().over(w))                   # ROW_NUMBER() OVER (...)
    .withColumn("prev_amount",
                F.lag("amount").over(w)))                 # LAG(amount) OVER (...)

enriched.show(truncate=False)


In [ ]:
# A framed window: moving average of amount over the current row and the two preceding
moving_w = (Window
    .partitionBy("customer_id")
    .orderBy("order_date")
    .rowsBetween(-2, 0))     # ROWS BETWEEN 2 PRECEDING AND CURRENT ROW

(orders
 .withColumn("avg_last_3",
             F.round(F.avg("amount").over(moving_w), 2))
 .show(truncate=False))


In [ ]:
# The exact same computation written as Spark SQL. Identical query plan,
# identical result — DataFrame API and SQL share the Catalyst optimizer.
orders.createOrReplaceTempView("orders")

spark.sql('''
    SELECT customer_id, order_date, amount,
           ROUND(SUM(amount)  OVER (PARTITION BY customer_id ORDER BY order_date), 2) AS running_total,
           ROW_NUMBER()       OVER (PARTITION BY customer_id ORDER BY order_date)     AS order_number,
           LAG(amount)        OVER (PARTITION BY customer_id ORDER BY order_date)     AS prev_amount,
           ROUND(AVG(amount)  OVER (PARTITION BY customer_id ORDER BY order_date
                                    ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2)     AS avg_last_3
    FROM orders
    ORDER BY customer_id, order_date
''').show(truncate=False)


### 1f. Spark SQL — three common patterns

Slide 34 of the deck showed three query shapes you will write over and over. Here they are running against the HealthPulse data you just loaded.


In [ ]:
# Register the DataFrames as temporary SQL views
patients_h01.createOrReplaceTempView("patients")
claims.createOrReplaceTempView("claims")


In [ ]:
# Pattern 1: Join + aggregate + filter + limit
# (The classic "top N" pattern.)
spark.sql('''
    SELECT p.dept,
           COUNT(*)                        AS n_claims,
           ROUND(SUM(c.billed_amount), 2)  AS total_billed,
           ROUND(AVG(c.billed_amount), 2)  AS avg_billed
    FROM patients p
    JOIN claims   c ON p.patient_id = c.patient_id
    GROUP BY p.dept
    HAVING SUM(c.billed_amount) > 1000
    ORDER BY total_billed DESC
    LIMIT 10
''').show(truncate=False)


In [ ]:
# Pattern 2: CTE + window function — top-N per group
# Find the 3 most expensive claims per department. The CTE computes a
# per-department rank; the outer query keeps only the top three.
spark.sql('''
    WITH ranked AS (
        SELECT p.dept, c.claim_id, c.billed_amount,
               ROW_NUMBER() OVER (
                   PARTITION BY p.dept
                   ORDER BY c.billed_amount DESC
               ) AS rn
        FROM   patients p
        JOIN   claims   c ON p.patient_id = c.patient_id
    )
    SELECT dept, claim_id, billed_amount
    FROM   ranked
    WHERE  rn <= 3
    ORDER  BY dept, billed_amount DESC
''').show(truncate=False)


In [ ]:
# Pattern 3: CASE + date function + subquery
# Tag each claim by size bucket, and keep only claims from patients in
# departments that have AT LEAST 5 claims overall.
spark.sql('''
    SELECT c.patient_id,
           c.billed_amount,
           YEAR(c.service_date) AS service_year,
           CASE
               WHEN c.billed_amount > 10000 THEN 'large'
               WHEN c.billed_amount > 2500  THEN 'medium'
               ELSE                             'small'
           END AS size_bucket
    FROM   claims c
    WHERE  c.patient_id IN (
        SELECT p.patient_id
        FROM   patients p
        WHERE  p.dept IN (
            SELECT p2.dept
            FROM   patients p2
            JOIN   claims   c2 ON p2.patient_id = c2.patient_id
            GROUP  BY p2.dept
            HAVING COUNT(*) >= 5
        )
    )
    ORDER BY c.billed_amount DESC
    LIMIT 10
''').show(truncate=False)


**How much SQL does Spark SQL support?** Spark SQL 3.5 targets ANSI SQL:2011/2016 for analytic queries — every JOIN flavor, window functions, CTEs (including recursive), scalar & correlated subqueries, `UNION`/`INTERSECT`/`EXCEPT`, `CASE`, date/time functions. Strict ANSI semantics are on by default (`spark.sql.ansi.enabled=true`). What it does NOT do: PL/pgSQL, triggers, stored procedures, foreign data wrappers, transactional DDL. It's an **analytic** engine (same class as Snowflake / BigQuery / Redshift), not a transactional RDBMS.


### 1g. Lazy evaluation, `.explain()`, and the DAG

Slide 33 introduced DAGs; slide 33b showed a real one from the Spark UI. Let's recreate that experience.

1. Build a chain of transformations. **Nothing happens** yet — transformations are lazy.
2. Call `.explain()`. You will see the physical plan, which is literally the text form of the DAG.
3. Call `.count()` (an action). **Now** the Driver assembles the DAG, schedules the tasks, and dispatches them to the Executors.
4. Open [http://localhost:4040/SQL/](http://localhost:4040/SQL/) in a browser tab. You will see the same plan rendered as a visual graph — the screenshot from slide 33b.


In [ ]:
# Build a chain of transformations. Nothing executes.
pipeline = (patients_h01
    .filter(F.col("age") >= 30)
    .withColumn("age_decade", (F.floor(F.col("age") / 10) * 10).cast("int"))
    .groupBy("dept", "age_decade")
    .agg(F.count("*").alias("count"))
    .orderBy("dept", "age_decade"))

print("=== Physical plan (this is the DAG, as text) ===")
pipeline.explain()
print("\n↑ No data has been processed yet. Spark just told us what it WOULD do.")


In [ ]:
# Now an action — NOW the DAG actually executes.
print(f"Result rows: {pipeline.count()}")
pipeline.show(10)

print()
print("Go look at the DAG you just triggered:")
print("   http://localhost:4040/SQL/   →  click the most recent query")
print("You will see Scan → Filter → Project → HashAggregate → Exchange → HashAggregate → Sort.")


---

## Part 2: The "T" in ELT — A Complete Three-Stage Pipeline

**Maps to lecture §5 (slides 36–40).**

Most real pipelines follow the same shape: three jobs, each reading from the previous zone and writing to the next. Here we build that pipeline end to end on a small DataFrame, showing you the data at each stage. This is exactly the pattern the HealthPulse and GreenRoute scenarios in Parts 4 and 5 follow at production scale.


In [ ]:
# We'll do the pipeline on a small synthetic orders DataFrame so you can SEE
# the data change at each stage (not just trust the pipeline).
from pyspark.sql.types import StringType, DoubleType, StructType, StructField

raw_input = spark.createDataFrame([
    ("1001", "45.50",  "2025-03-12T10:15", "fl",     "prod_a", "electronics"),
    ("1002", "89.00",  "2025-03-12T10:17", "FL",     "prod_b", "apparel"),
    ("1002", None,     "2025-03-12T10:17", "FL",     "prod_b", "apparel"),     # dup + null amount
    ("1003", "120",    "2025-03-12T10:22", "Fl",     "prod_c", "electronics"),
    ("1004", "250",    "2025-03-12T11:05", "FL",     "prod_a", "electronics"),
    ("1005", "  30",   "2025-03-12T11:18", "fl",     "prod_d", "home"),
], ["order_id", "amount", "order_ts", "region", "product_id", "category"])

print("=== Raw input (as it arrives from the source system) ===")
raw_input.show(truncate=False)
print("Look at the mess: `amount` is string with nulls and whitespace; `region` has four case variants;")
print("`order_ts` is a string; there's a duplicate order_id 1002. This is why curated zones exist.")


### 2a. Ingestion → Landing

The landing zone's job is to durably capture the raw file **without transformation**. Schema-on-read, warts and all. This is what you'd write if you were building a production ingest job.


In [ ]:
LAKE = "/home/jovyan/data/week09_pipeline_demo"
import shutil
shutil.rmtree(LAKE, ignore_errors=True)   # start clean for this demo

# Write exactly what we received, as Parquet. No transformations.
(raw_input.write
     .mode("overwrite")
     .parquet(f"{LAKE}/landing/orders"))

# Read it back to confirm
landed = spark.read.parquet(f"{LAKE}/landing/orders")
print(f"=== /landing/orders — {landed.count()} rows ===")
landed.show(truncate=False)


### 2b. Curation → Curated

The curated zone is the **trusted** copy. Here's where cleanup lives: cast types, normalize strings, drop duplicates, derive partition columns.

Follow what `upper(col("region"))` actually does below — the four raw case variations (`fl` / `FL` / `FL` / `Fl`) all collapse to `FL`. Nothing smarter is happening; it's just a string `upper()`.


In [ ]:
from pyspark.sql.functions import col, to_date, upper, trim

curated = (spark.read.parquet(f"{LAKE}/landing/orders")
    .filter(col("amount").isNotNull())                            # drop rows with null amount
    .withColumn("amount", trim(col("amount")).cast("decimal(12,2)"))  # cast string -> typed decimal
    .withColumn("order_date", to_date(col("order_ts")))           # derive DATE partition key
    .withColumn("region", upper(col("region")))                   # normalize case
    .dropDuplicates(["order_id"])                                  # one row per order
    .select("order_id", "amount", "order_date", "region", "product_id", "category"))

(curated.write
    .mode("overwrite")
    .partitionBy("order_date")
    .parquet(f"{LAKE}/curated/orders"))

print("=== /curated/orders (partitioned by order_date) ===")
# Read back to confirm: typed, deduped, uppercased, partitioned
read_back = spark.read.parquet(f"{LAKE}/curated/orders")
read_back.printSchema()
read_back.show(truncate=False)


### 2c. Analytics → Analytics

The analytics zone holds **pre-aggregated, narrow** datasets tuned for specific BI questions. Dashboards read from here. If the question changes, you write a new analytics table — you don't mutate this one.


In [ ]:
# A small dimension table to join against (products → category rollup)
products = spark.createDataFrame([
    ("prod_a", "electronics", 1.0),
    ("prod_b", "apparel",     0.8),
    ("prod_c", "electronics", 1.2),
    ("prod_d", "home",        0.6),
], ["product_id", "category_std", "weight"])

# Analytics job: join, filter, aggregate, partition-write
from pyspark.sql.functions import sum as spark_sum, count as sql_count, avg, broadcast

summary = (
    spark.read.parquet(f"{LAKE}/curated/orders")
        .join(broadcast(products), "product_id")
        .filter(col("order_date") >= "2025-01-01")
        .groupBy("category", "region")
        .agg(spark_sum("amount").alias("revenue"),
             sql_count("*").alias("n_orders"),
             avg("amount").alias("avg_amount"))
)

(summary.write
    .mode("overwrite")
    .partitionBy("region")
    .parquet(f"{LAKE}/analytics/revenue"))

print("=== /analytics/revenue (partitioned by region) ===")
spark.read.parquet(f"{LAKE}/analytics/revenue").show(truncate=False)


### 2d. What landed on disk

Each stage is a real Parquet directory tree. Notice how the curated and analytics zones are partitioned — every partition key becomes a subdirectory. That's what makes **partition pruning** work in Part 3.


In [ ]:
import os
for root, dirs, files in os.walk(LAKE):
    level = root.replace(LAKE, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root) or LAKE}/")
    if level <= 3:
        for f in sorted(files):
            if f.endswith(".parquet") or f.endswith(".crc") or f == "_SUCCESS":
                print(f"{indent}  {f}")


---

## Part 3: Optimization (Optional Deep Dive)

**Maps to lecture §6 — every slide in §6 is marked Deep Dive · Optional.**

Skip this section if you're short on time; the HealthPulse and GreenRoute scenarios in Parts 4 and 5 do not depend on it. Come back here when you want to understand *why* a pipeline is slow and what to do about it.


### 3a. Caching — time the difference

When a DataFrame is reused by multiple actions, caching avoids recomputing it every time. The difference is visible in wall-clock time.


In [ ]:
# Read the curated HealthPulse patient data we'll load in Part 4, but load it here.
# (We'll use the raw CSV so the timing is honest — no Parquet cheating.)
all_h = spark.read.csv(
    "/home/jovyan/data/healthpulse/patient-records/hospital_01_patients.csv",
    header=True, inferSchema=True,
)

import time

# Uncached — each action re-reads from disk
t0 = time.time()
for _ in range(3):
    all_h.filter(F.col("dept") == "Cardiology").count()
uncached_sec = time.time() - t0
print(f"3 × count(), uncached: {uncached_sec:.2f}s")


In [ ]:
# Cache and materialize
all_h.cache()
all_h.count()   # this action materializes the cache

t0 = time.time()
for _ in range(3):
    all_h.filter(F.col("dept") == "Cardiology").count()
cached_sec = time.time() - t0
print(f"3 × count(), cached:   {cached_sec:.2f}s")
print(f"Speedup: {uncached_sec / max(cached_sec, 0.001):.1f}×")

# Open http://localhost:4040/storage to SEE the cached blocks.
all_h.unpersist()


### 3b. Reading a query plan with `explain(True)`

`.explain(True)` dumps **all four stages** of the Catalyst optimizer: parsed → analyzed → optimized → physical. For day-to-day debugging, the physical plan (at the bottom) is the one you care about.


In [ ]:
# A moderately complex query
q = (
    patients_h01
        .filter(F.col("age") > 50)
        .join(claims, on="patient_id", how="inner")
        .groupBy("dept")
        .agg(F.count("*").alias("n"), F.round(F.sum("billed_amount"), 2).alias("total"))
        .filter(F.col("n") > 10)
        .orderBy(F.desc("total"))
)

# Physical plan only — terse, readable
q.explain()


### 3c. Partition pruning

Partitioned Parquet is the single biggest performance lever available. When the partition column appears in a filter, Spark **skips whole directories** — orders of magnitude less I/O.


In [ ]:
# We partitioned curated/orders by order_date in Part 2. Filter on it.
pruned = (spark.read.parquet(f"{LAKE}/curated/orders")
          .filter(F.col("order_date") == "2025-03-12"))

print("=== Physical plan (look for 'PartitionFilters') ===")
pruned.explain()
pruned.show(truncate=False)


### 3d. `repartition` vs `coalesce`

Both change the number of partitions, but with different costs.


In [ ]:
# Current partition count
print(f"curated/orders partitions: {read_back.rdd.getNumPartitions()}")

# repartition(n) — full shuffle. Use before a skewed join or when you need
# perfectly balanced partitions.
print(f"After repartition(6):       {read_back.repartition(6).rdd.getNumPartitions()}")

# coalesce(n) — no shuffle. Merges adjacent partitions. Cheap, but the output
# partitions may be uneven. Use right before .write() to control output file count.
print(f"After coalesce(2):          {read_back.coalesce(2).rdd.getNumPartitions()}")


---

## Part 4: HealthPulse — Multi-Source Harmonization

**Maps to lecture §7 (slides for HealthPulse Medical Analytics).**

HealthPulse aggregates four hospital CSVs (with slightly different schemas), IoT device-reading JSON, insurance claims CSV, and lab result JSON. The deliverable is a **curated Parquet data mart** with unified patients, billing analytics, anomaly flags, and critical-lab summaries.

### 4.1 Load all four hospital files


In [ ]:
h01 = spark.read.csv("/home/jovyan/data/healthpulse/patient-records/hospital_01_patients.csv", header=True, inferSchema=True)
h02 = spark.read.csv("/home/jovyan/data/healthpulse/patient-records/hospital_02_patients.csv", header=True, inferSchema=True)
h03 = spark.read.csv("/home/jovyan/data/healthpulse/patient-records/hospital_03_patients.csv", header=True, inferSchema=True)
h04 = spark.read.csv("/home/jovyan/data/healthpulse/patient-records/hospital_04_patients.csv", header=True, inferSchema=True)

# Schemas differ — that's why we need to harmonize
for name, df in [("H01", h01), ("H02", h02), ("H03", h03), ("H04", h04)]:
    print(f"{name}: {df.count()} rows, columns = {df.columns}")


Notice the department column has a different name in each file. This is the normal reality when several source systems feed a data lake.

### 4.2 Harmonize schemas

We standardize every hospital DataFrame to the same columns: `patient_id`, `name`, `age`, `gender`, `department`, `hospital_id`. Then we `union` them into one unified patients table.

**Important:** hospital_01's CSV has `dob`, not `age`. So before harmonizing we derive `age` for it.


In [ ]:
# Helper: derive age from dob for hospital files that have dob but no age
def add_age_from_dob(df):
    if "age" in df.columns:
        return df
    if "dob" in df.columns:
        return df.withColumn(
            "age",
            F.floor(F.datediff(F.current_date(), F.col("dob")) / 365.25).cast("int"),
        )
    raise ValueError(f"No age or dob column in {df.columns}")

h01 = add_age_from_dob(h01)
h02 = add_age_from_dob(h02)
h03 = add_age_from_dob(h03)
h04 = add_age_from_dob(h04)


def harmonize(df, hospital_id, dept_col):
    '''Standardize one hospital DataFrame to the common schema.

    Note: the real Week 9 hospital CSVs do NOT have a `gender` column;
    they do have `patient_id, name, age (derived), dept variant, cost`.
    We project to the intersection that every hospital exposes.
    '''
    return (df.select(
        F.col("patient_id"),
        F.col("name"),
        F.col("age"),
        F.col(dept_col).alias("department"),
        F.col("cost").alias("admission_cost"),
    ).withColumn("hospital_id", F.lit(hospital_id)))


# Detect the department column in each file (varies by hospital)
def dept_col_of(df):
    for c in df.columns:
        if "dept" in c.lower() or "department" in c.lower():
            return c
    raise ValueError("No dept/department column")


h01_std = harmonize(h01, "hospital_01", dept_col_of(h01))
h02_std = harmonize(h02, "hospital_02", dept_col_of(h02))
h03_std = harmonize(h03, "hospital_03", dept_col_of(h03))
h04_std = harmonize(h04, "hospital_04", dept_col_of(h04))

all_patients = h01_std.union(h02_std).union(h03_std).union(h04_std)
print(f"Unified patients: {all_patients.count():,} rows")
all_patients.groupBy("hospital_id").count().orderBy("hospital_id").show()


### 4.3 Enrich with insurance claims


In [ ]:
claims = spark.read.csv(
    "/home/jovyan/data/healthpulse/insurance-claims/insurance_claims.csv",
    header=True, inferSchema=True,
)
print(f"Claims: {claims.count():,} rows")

# Billing analytics per patient — use the real columns: billed_amount, status.
billing = (
    all_patients.join(claims, on="patient_id", how="inner")
        .groupBy("patient_id", "name", "hospital_id", "department")
        .agg(
            F.count("*").alias("claim_count"),
            F.round(F.sum("billed_amount"), 2).alias("total_billed"),
            F.round(F.avg("billed_amount"), 2).alias("avg_claim_amount"),
            F.round(
                F.sum(F.when(F.col("status") == "approved", 1).otherwise(0))
                / F.count("*") * 100, 1
            ).alias("approval_rate_pct"),
        )
)
print("\n=== Top 10 patients by total billed ===")
billing.orderBy(F.desc("total_billed")).show(10, truncate=False)


### 4.4 Device-reading anomaly detection (nested JSON)

The device readings JSON has a nested `reading` struct — we flatten it and flag values outside clinical normal ranges.


In [ ]:
device = spark.read.json("/home/jovyan/data/healthpulse/device-readings/device_readings.json")
print(f"Device readings: {device.count():,} rows")
device.printSchema()
device.show(3, truncate=False)


In [ ]:
# Flatten the nested `reading` struct: reading.heart_rate, reading.spo2, etc.
# become top-level columns.
flat = device.select(
    "patient_id",
    "device_type",
    "timestamp",
    "hospital_id",
    "reading.*",
)
flat.printSchema()

# Figure out which numeric columns the data actually has
numeric_cols = [c for c, dt in flat.dtypes if dt in ("double", "float", "int", "bigint") and c not in ("patient_id", "hospital_id")]
print(f"Numeric vital-sign columns: {numeric_cols}")


In [ ]:
# Flag anomalies dynamically based on what columns exist.
# Clinical thresholds: heart rate > 100 bpm, SpO2 < 92%, systolic BP > 160 mmHg.
flagged = flat
rules = [
    (["heart", "hr", "bpm", "pulse"], "high_heart_rate",   lambda c: c > 100),
    (["spo2", "oxygen",  "o2"],       "low_spo2",          lambda c: c < 92),
    (["systolic", "sbp", "bp_sys"],   "high_bp_systolic",  lambda c: c > 160),
]

applied = []
for keywords, label, pred in rules:
    matches = [c for c in numeric_cols if any(k in c.lower() for k in keywords)]
    if matches:
        flagged = flagged.withColumn(label, pred(F.col(matches[0])).cast("int"))
        applied.append((label, matches[0]))

print("Rules applied:")
for label, src in applied:
    print(f"  {label:<22s}  based on column `{src}`")

if applied:
    agg_cols = [F.count("*").alias("total_readings")] + [F.sum(lab).alias(lab) for lab, _ in applied]
    flagged.groupBy("hospital_id").agg(*agg_cols).orderBy("hospital_id").show(truncate=False)


### 4.5 Lab-result analysis


In [ ]:
labs = spark.read.json("/home/jovyan/data/healthpulse/lab-results/lab_results.json")
print(f"Lab results: {labs.count():,} rows")
labs.printSchema()

# Find the flag / status column
flag_col = next((c for c in labs.columns
                 if "flag" in c.lower() or "status" in c.lower() or "critical" in c.lower()), None)
print(f"Flag column: {flag_col}")
if flag_col:
    labs.groupBy(flag_col).count().orderBy(F.desc("count")).show()


In [ ]:
# Filter to critical results and attach hospital context by joining on patient_id
if flag_col:
    critical = labs.filter(F.lower(F.col(flag_col)).contains("critical"))
    critical_with_hospital = critical.join(
        all_patients.select("patient_id", "hospital_id", "department"),
        on="patient_id", how="inner",
    )
    # Critical results per hospital per test type (if test_name column exists)
    test_col = next((c for c in labs.columns if "test" in c.lower() and "name" in c.lower()), None)
    if test_col is None:
        test_col = next((c for c in labs.columns if "test" in c.lower() or "type" in c.lower()), None)
    if test_col:
        (critical_with_hospital
            .groupBy("hospital_id", test_col)
            .agg(F.count("*").alias("critical_count"))
            .orderBy("hospital_id", F.desc("critical_count"))
            .show(20, truncate=False))


### 4.6 Write the HealthPulse curated zone


In [ ]:
HP_OUT = "/home/jovyan/data/healthpulse/curated"
import shutil
shutil.rmtree(HP_OUT, ignore_errors=True)

(all_patients.write.mode("overwrite")
    .partitionBy("hospital_id")
    .parquet(f"{HP_OUT}/unified_patients"))

(billing.write.mode("overwrite")
    .parquet(f"{HP_OUT}/patient_billing"))

# Show directory shape
print("=== HealthPulse curated zone ===")
for root, dirs, files in os.walk(HP_OUT):
    level = root.replace(HP_OUT, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root) or HP_OUT}/")
    shown = sorted(files)[:4]
    for f in shown:
        print(f"{indent}  {f}")
    if len(files) > 4:
        print(f"{indent}  ... and {len(files) - 4} more")

# Size comparison: CSV inputs vs Parquet output
def dir_size(p):
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(p) for f in fs)

csv_size = sum(os.path.getsize(f"/home/jovyan/data/healthpulse/patient-records/hospital_{i:02d}_patients.csv")
               for i in range(1, 5))
pq_size  = dir_size(f"{HP_OUT}/unified_patients")

print()
print(f"CSV input (4 files):        {csv_size / 1024 / 1024:.2f} MB")
print(f"Parquet output (partitioned): {pq_size / 1024 / 1024:.2f} MB")
if pq_size > 0:
    print(f"Compression ratio:           {csv_size / pq_size:.1f}×")


---

## Part 5: GreenRoute — Trip Segmentation

**Maps to lecture §8 (slides for GreenRoute Logistics Analytics).**

GreenRoute tracks a fleet of delivery trucks via GPS telemetry. The core challenge is **time-series segmentation**: the raw GPS data is a stream of points; we need to group them into trips and compute per-trip metrics. That's a textbook use case for window functions.

### 5.1 Read the partitioned GPS data (Avro)

We downloaded 72 GPS partitions (3 days × 24 hours). A wildcard path reads them all at once — Spark parallelizes the read across Executors.


In [ ]:
import glob
gps_files = glob.glob("/home/jovyan/data/greenroute/gps-telemetry/day_*/hour_*/gps_data.avro")
print(f"GPS Avro partitions on disk: {len(gps_files)}")

gps = spark.read.format("avro").load(
    "/home/jovyan/data/greenroute/gps-telemetry/day_*/hour_*/gps_data.avro"
)
print(f"GPS records loaded:          {gps.count():,}")
print(f"Spark partitions (tasks):    {gps.rdd.getNumPartitions()}")
gps.printSchema()
gps.show(3, truncate=False)


### 5.2 Trip segmentation with window functions

A trip is a contiguous run of non-stopped GPS readings for one truck. The trick: use a window partitioned by truck and ordered by time, flag each stop as a `1`, then a running sum of the stop flag becomes a **segment ID** that increments after every stop.


In [ ]:
from pyspark.sql.window import Window

# Find the right column names (real data has variations)
TRUCK = next((c for c in gps.columns if "truck" in c.lower() or "vehicle" in c.lower()), "truck_id")
SPEED = next((c for c in gps.columns if "speed" in c.lower()), "speed")
TIME  = next((c for c in gps.columns if "timestamp" in c.lower() or "time" in c.lower() or "ts" in c.lower()), "timestamp")
print(f"Using: truck={TRUCK}, speed={SPEED}, time={TIME}")

truck_w = Window.partitionBy(TRUCK).orderBy(TIME)

gps_seg = (gps
    .withColumn("is_stop",
                F.when(F.col(SPEED) < 2, 1).otherwise(0))
    .withColumn("trip_segment",
                F.sum("is_stop").over(truck_w)))

gps_seg.select(TRUCK, TIME, SPEED, "is_stop", "trip_segment").orderBy(TRUCK, TIME).show(15, truncate=False)


In [ ]:
# Aggregate per segment: start, end, duration, avg speed, max speed
trips = (gps_seg
    .filter(F.col("is_stop") == 0)
    .groupBy(TRUCK, "trip_segment")
    .agg(F.min(TIME).alias("trip_start"),
         F.max(TIME).alias("trip_end"),
         F.count("*").alias("gps_points"),
         F.round(F.avg(SPEED), 1).alias("avg_speed"),
         F.round(F.max(SPEED), 1).alias("max_speed"))
    # GPS timestamps are ISO-8601 with a trailing Z (UTC). Parse with an explicit
    # format so unix_timestamp returns a real long — otherwise it silently yields NULL
    # and duration_minutes becomes NULL.
    .withColumn("duration_minutes",
                F.round(
                    (F.unix_timestamp(F.col("trip_end"),   "yyyy-MM-dd'T'HH:mm:ss'Z'")
                     - F.unix_timestamp(F.col("trip_start"), "yyyy-MM-dd'T'HH:mm:ss'Z'"))
                    / 60, 1)))

print(f"Trip segments identified: {trips.count():,}")
trips.orderBy(TRUCK, "trip_start").show(15, truncate=False)


### 5.3 Delivery enrichment — explode nested arrays


In [ ]:
from pyspark.sql.types import ArrayType

deliveries = spark.read.json("/home/jovyan/data/greenroute/delivery-receipts/delivery_receipts.json")
print(f"Delivery receipts: {deliveries.count():,} rows")
deliveries.printSchema()

array_cols = [c for c, dt in deliveries.dtypes if dt.startswith("array")]
print(f"Array columns to explode: {array_cols}")


In [ ]:
if array_cols:
    items_col = array_cols[0]
    exploded = deliveries.withColumn("item", F.explode(F.col(items_col)))
    print(f"After exploding `{items_col}`:")
    exploded.printSchema()
    exploded.show(5, truncate=False)


### 5.4 Fuel efficiency — window LAG for previous odometer


In [ ]:
fuel = spark.read.csv(
    "/home/jovyan/data/greenroute/fuel-logs/fuel_logs.csv",
    header=True, inferSchema=True,
)
print(f"Fuel logs: {fuel.count():,} rows")
fuel.printSchema()

# Detect columns
ft  = next((c for c in fuel.columns if "truck" in c.lower() or "vehicle" in c.lower()), None)
fg  = next((c for c in fuel.columns if "gallon" in c.lower() or "fuel" in c.lower() or "liter" in c.lower()), None)
fod = next((c for c in fuel.columns if "odomet" in c.lower() or "mile" in c.lower() or "km" in c.lower()), None)
ftm = next((c for c in fuel.columns if "time" in c.lower() or "date" in c.lower()), None)
print(f"truck={ft}  gallons={fg}  odometer={fod}  time={ftm}")


In [ ]:
if ft and fg and fod and ftm:
    fw = Window.partitionBy(ft).orderBy(ftm)
    per_fill = (fuel
        .withColumn("prev_odo", F.lag(fod).over(fw))
        .withColumn("miles_driven", F.col(fod) - F.col("prev_odo"))
        .withColumn("mpg", F.round(F.col("miles_driven") / F.col(fg), 2))
        .filter(F.col("prev_odo").isNotNull()))

    per_truck = (per_fill.groupBy(ft)
        .agg(F.count("*").alias("fill_ups"),
             F.round(F.sum("miles_driven"), 1).alias("total_miles"),
             F.round(F.sum(fg), 1).alias("total_gallons"),
             F.round(F.avg("mpg"), 2).alias("avg_mpg"))
        .orderBy(F.desc("avg_mpg")))
    print("=== Fuel efficiency per truck (best → worst) ===")
    per_truck.show(20, truncate=False)


### 5.5 Weather broadcast join


In [ ]:
weather = spark.read.json("/home/jovyan/data/greenroute/weather/weather_data.json")
print(f"Weather: {weather.count():,} rows — small enough to broadcast")
weather.printSchema()

cond_col = next((c for c in weather.columns if "condition" in c.lower() or "type" in c.lower() or "weather" in c.lower()), None)
wtm_col  = next((c for c in weather.columns if "time" in c.lower() or "date" in c.lower() or "hour" in c.lower()), None)
dtm_col  = next((c for c in deliveries.columns if "time" in c.lower() or "date" in c.lower()), None)
print(f"weather_condition={cond_col}  weather_time={wtm_col}  delivery_time={dtm_col}")


In [ ]:
if cond_col and wtm_col and dtm_col:
    weather_prep = weather.withColumn("join_date", F.to_date(F.col(wtm_col)))
    deliv_prep   = deliveries.withColumn("join_date", F.to_date(F.col(dtm_col)))

    # For each day, keep the most frequent weather condition
    dw = Window.partitionBy("join_date").orderBy(F.desc("n"))
    dominant = (weather_prep.groupBy("join_date", cond_col)
        .agg(F.count("*").alias("n"))
        .withColumn("rank", F.row_number().over(dw))
        .filter(F.col("rank") == 1)
        .drop("n", "rank"))

    # Broadcast join — small dominant-weather DF, big deliveries DF
    joined = deliv_prep.join(F.broadcast(dominant), on="join_date", how="left")

    print("=== Deliveries by daily weather condition ===")
    joined.groupBy(cond_col).agg(F.count("*").alias("delivery_count")).orderBy(F.desc("delivery_count")).show()

    print()
    print("=== Physical plan (BroadcastHashJoin visible) ===")
    joined.explain()


### 5.6 Write the GreenRoute curated zone


In [ ]:
GR_OUT = "/home/jovyan/data/greenroute/curated"
import shutil
shutil.rmtree(GR_OUT, ignore_errors=True)

(trips.write.mode("overwrite")
    .partitionBy(TRUCK)
    .parquet(f"{GR_OUT}/trip_segments"))

if ft and fg and fod and ftm:
    (per_truck.write.mode("overwrite")
        .parquet(f"{GR_OUT}/fuel_efficiency"))

# Size comparison: Avro input vs Parquet output
avro_size = sum(os.path.getsize(f) for f in gps_files)
trip_pq_size = dir_size(f"{GR_OUT}/trip_segments")

print(f"GPS Avro input (3 days):   {avro_size / 1024 / 1024:.2f} MB")
print(f"Trip Parquet output:       {trip_pq_size / 1024 / 1024:.2f} MB")
print("(Output is aggregated, so size reduction reflects aggregation + compression.)")


---

## Part 6: Reflection Questions & Cleanup

The questions below ask you to connect the hands-on work above to the architectural concepts from the lecture. Try to answer them without looking back at the slides first — then check yourself.

---

**Question 1 — Format choices.** You read CSV, JSON, Avro, and Parquet in this notebook. Order them from fastest to slowest to read at scale, and explain why. When would you deliberately pick the slower one?

---

**Question 2 — The DAG you saw.** In Part 1g you ran `.explain()` and pointed your browser at `http://localhost:4040/SQL/`. What operators did you see, and which of them correspond to shuffle boundaries? Where did `HashAggregate` appear twice and why?

---

**Question 3 — Broadcast vs shuffle join.** In Part 5.5 we broadcast-joined 3,360 weather rows against many thousands of deliveries. What would happen at 100 GB of weather data instead? What would the physical plan look like, and roughly how expensive would it be?

---

**Question 4 — Schema harmonization or schema registry.** Part 4.2 had to custom-rename columns across four hospitals. How would an Avro + schema-registry setup (Week 8) have prevented that work? Sketch the two-sentence pitch you'd give your CTO.

---

**Question 5 — Pipeline promotion.** In Part 2 you wrote a three-stage pipeline into `/home/jovyan/data/week09_pipeline_demo/{landing,curated,analytics}`. If you had to promote that pipeline to a production cluster with 500× more data, which two optimizations from Part 3 would you enable first, and why?


## Summary

You built two end-to-end Spark data pipelines that mirror real production workloads, wired each concept to a specific slide section, and saw the DAG that Spark's Catalyst optimizer created on your behalf.

| | HealthPulse (Part 4) | GreenRoute (Part 5) |
|---|---|---|
| Industry | Healthcare | Logistics |
| Raw inputs | 4× hospital CSV, IoT JSON, claims CSV, lab JSON | Partitioned GPS Avro, delivery JSON, weather JSON, fuel CSV |
| Core challenge | Schema variation across source systems | Time-series segmentation |
| Key techniques | Harmonization, joins, anomaly flags, aggregation | Window functions, explode, broadcast join, per-truck MPG |
| Output | Parquet, partitioned by `hospital_id` | Parquet, partitioned by `truck_id` |

**Five ideas to take away:**

1. **Laziness is a feature.** Spark waits for an action so it can see the whole DAG and rewrite it.
2. **Schema harmonization is 80% of real Transform code.** Make peace with it.
3. **Parquet + partitioning + predicate pushdown is the biggest performance lever** available to you without touching cluster config.
4. **Broadcast any join where one side is < ~100 MB.** It eliminates the shuffle entirely.
5. **Window functions = SQL `OVER(...)`.** If you can write it in Postgres, you can write it in Spark.

Next week we add the Stream layer with Kafka — the same Transform pipelines, but processing real-time events as they arrive.


In [ ]:
# Clean up: stop the SparkSession (releases the Executors on the workers)
spark.stop()
print("SparkSession stopped.")
print("To shut down the whole cluster: docker compose down")
